# Trabajo Practico Grupal
## Laboratorio de Metodos Cuantitativos Aplicados a la Gestión · FCE UBA · 1C 2026

---

### Sobre la empresa

**General Cereals S.A.** es una empresa argentina de capitales nacionales fundada en 1994, con sede en Villa Adelina, Provincia de Buenos Aires. Se dedica a la elaboracion de cereales para el desayuno bajo su marca principal **NUTRI FOODS**, ofreciendo productos como copos azucarados, anillos, bolitas de cereal, granola, avena y almohaditas, tanto en presentaciones hogar y familiar como a granel para clientes industriales. En 2014 fue adquirida por el **Grupo Georgalos**. Ademas de sus marcas propias, produce cereales como insumo para terceros (barras de cereal, chocolates, helados, alfajores) y exporta a mas de 10 paises.

---

### Nuestra pregunta de investigacion

> **¿Los productos que recibieron octogonos de advertencia por la Ley 27.642 de etiquetado frontal vendieron menos unidades que los que no los recibieron, teniendo en cuenta que la inflacion ya de por si redujo el consumo general?**

### ¿Por que esta pregunta importa?

En julio de 2022 entro en vigor en Argentina la **Ley 27.642 de Etiquetado Frontal**. Esta ley obliga a los alimentos con exceso de azucar, sodio o grasas a llevar octogonos negros de advertencia en el frente del envase, con el objetivo de que los consumidores elijan productos mas saludables.

El problema para medir ese efecto es que **en el mismo periodo Argentina atraveso una inflacion muy alta**, que redujo el consumo en general. Si las ventas de un producto bajaron, ¿fue por el octogono o por la inflacion?

General Cereals nos da una ventaja unica para responder esto: vende productos **CON octogono** (cereales azucarados como copos, bolitas, anillos) y productos **SIN octogono** (avena, granola, salvado) dentro de la misma empresa. Si ambos grupos sufrieron la misma inflacion pero uno cayo mas que el otro, esa diferencia es atribuible al etiquetado.

---
**Estructura del notebook:**
- Seccion 3.1 — Carga y exploracion inicial *(esta seccion)*
- Seccion 3.2 — Pregunta de investigacion
- Seccion 3.3 — Transformaciones y resumen estadistico
- Seccion 3.4 — Analisis grafico
- Seccion 3.5 — Aplicacion de conceptos de la materia

---
# Seccion 3.1 — Carga y exploracion inicial del dataset

### ¿Que hacemos en esta seccion y por que?

Antes de responder cualquier pregunta con datos, hay que conocer el dataset a fondo. Igual que antes de cocinar necesitas revisar los ingredientes, antes de analizar necesitas entender con que datos contas.

En esta seccion vamos a:
1. **Importar** las librerias necesarias
2. **Cargar** el dataset con `pd.read_csv()`
3. **Ver** las primeras y ultimas filas
4. **Entender la estructura**: tipos de datos, columnas, dimensiones
5. **Verificar la calidad**: nulos, duplicados, valores anomalos
6. **Explorar** los productos y la cobertura temporal
7. **Hacer un primer grafico** para visualizar los datos

Cada hallazgo aqui va a justificar las decisiones que tomamos en las secciones siguientes.


## 1.1 — Importacion de librerias

### ¿Que son las librerias?

Python por defecto no sabe trabajar con tablas de datos ni hacer graficos. Las **librerias** son paquetes de herramientas adicionales que le dan esas capacidades. Hay que importarlas al principio del notebook para poder usarlas despues.

Las que vamos a usar en todo el trabajo son:

| Libreria | Alias | Para que la usamos |
|---|---|---|
| `pandas` | `pd` | Cargar, filtrar, agrupar y transformar la tabla de ventas |
| `numpy` | `np` | Operaciones matematicas, algebra lineal y calculos numericos |
| `matplotlib.pyplot` | `plt` | Crear todos los graficos |
| `matplotlib.patches` | — | Elementos visuales adicionales (leyendas con color en graficos) |
| `seaborn` | `sns` | Graficos estadisticos mas elaborados |
| `sympy` | `sp` | Modelado simbolico de la regresion lineal (seccion 3.5) |
| `warnings` | — | Silenciar alertas menores que no afectan los resultados |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
import sympy as sp
import warnings
warnings.filterwarnings('ignore')

print('Librerias importadas correctamente.')

## 1.2 — Carga del dataset

### ¿Que es un DataFrame y como cargamos el archivo?

Un **DataFrame** es la estructura principal de pandas: una tabla con filas y columnas, similar a una hoja de Excel pero mucho mas potente para analisis.

Usamos `pd.read_csv()` para cargar el archivo. El dataset fue preprocesado previamente para que funcione correctamente con esta funcion. Los parametros que usamos son:

- **`sep=';'`**: indica que los campos estan separados por punto y coma. Usamos este separador en lugar de la coma porque algunos campos del dataset (como nombres de productos o localidades) contienen comas internas que confundirian a un lector estandar.
- **`encoding='utf-8-sig'`**: permite leer correctamente caracteres especiales del español (tildes, enye) que estan presentes en los nombres de productos y localidades.

In [ ]:
df = pd.read_csv('data/Dataset_Limpio.csv', sep=';', encoding='utf-8-sig')

print(f'Dataset cargado exitosamente.')
print(f'Filas (transacciones): {df.shape[0]:,}')
print(f'Columnas (atributos):  {df.shape[1]}')

## 1.3 — Primeras y ultimas filas: `.head()` y `.tail()`

### ¿Para que miramos las primeras y ultimas filas?

Es el primer contacto visual con los datos. Con `.head()` vemos las primeras 5 filas y con `.tail()` las ultimas 5. Esto nos permite verificar que la carga fue correcta, que los valores tienen sentido y que no hay problemas evidentes al inicio o al final del archivo.

Mostramos solo las **columnas mas relevantes para nuestra pregunta** en lugar de las 53 columnas del dataset completo, para que la tabla sea legible.

**Columnas que vamos a usar en el analisis:**

| Columna | Que contiene | Por que nos importa |
|---|---|---|
| `FECHA` | Fecha de la transaccion | Para separar ventas antes y despues de la ley (jul-2022) |
| `FORMULARIO` | Tipo de comprobante | Diferencia ventas reales (FCD) de devoluciones (NCD) |
| `DETALLE` | Nombre del producto | Para identificar cada producto especifico |
| `PRECIO` | Precio unitario | Para medir el contexto inflacionario |
| `CANTIDAD` | Unidades vendidas | **Variable principal**: lo que queremos comparar |
| `SUBRUBRO_BI` | Categoria del producto | Base para clasificar si lleva octogono o no |
| `MARCA_BI` | Marca | Para entender el mix de marcas propias y terceros |
| `AÑO` | Año de la transaccion | Para comparaciones anuales |

In [ ]:
COLS = ['FECHA', 'FORMULARIO', 'DETALLE', 'PRECIO',
        'CANTIDAD', 'SUBRUBRO_BI', 'MARCA_BI', 'AÑO']

print('=== PRIMERAS 5 FILAS ===')
print('(Transacciones mas antiguas del dataset)')
display(df[COLS].head())

print('\n=== ULTIMAS 5 FILAS ===')
print('(Transacciones mas recientes del dataset)')
display(df[COLS].tail())

## 1.4 — Estructura del dataset: `.info()`

### ¿Que nos dice `.info()`?

El metodo `.info()` es el primer diagnostico de salud de un dataset. En una sola llamada nos muestra:

- Cuantas filas tiene el dataset
- El nombre de cada columna
- Cuantos valores **no nulos** tiene cada columna (si este numero es menor que el total de filas, hay datos faltantes)
- El **tipo de dato** de cada columna: `object` = texto, `float64` = numero decimal, `int64` = numero entero

Esto es importante porque Python no puede hacer cuentas con texto: si una columna numerica aparece como `object`, hay que convertirla.

In [ ]:
print('=== INFORMACION GENERAL DEL DATASET ===')
df.info()

## 1.5 — Conversion de la columna FECHA

### ¿Por que necesitamos convertir la fecha?

Cuando `.info()` muestra que `FECHA` es de tipo `object` (texto), significa que Python la trata como una cadena de caracteres: '18/01/2022'. Con texto no podemos hacer operaciones de fecha como 'mostrame todo lo que sea despues del 1 de julio de 2022'.

Usamos `pd.to_datetime()` para convertirla a un tipo fecha real. El parametro `format='%d/%m/%Y'` le indica el formato: dia/mes/año.

Una vez convertida, creamos la columna `AÑO_MES` con `.dt.to_period('M')` para poder agrupar transacciones por mes en los graficos.

In [ ]:
df['FECHA'] = pd.to_datetime(df['FECHA'], format='%d/%m/%Y', errors='coerce')
df['AÑO_MES'] = df['FECHA'].dt.to_period('M')

print('Conversion exitosa.')
print(f'Tipo de FECHA ahora: {df["FECHA"].dtype}')
print(f'Ejemplo de AÑO_MES: {df["AÑO_MES"].iloc[0]}')

## 1.6 — Estadisticas descriptivas: `.describe()`

### ¿Que nos dice `.describe()`?

El metodo `.describe()` calcula automaticamente las estadisticas basicas de cada columna numerica:

- **count**: cuantos valores no nulos hay
- **mean**: promedio
- **std**: desviacion estandar (que tan dispersos estan los valores)
- **min / max**: valor minimo y maximo
- **25% / 50% / 75%**: percentiles (el 50% es la mediana: el valor del medio si ordenas todos los datos)

Para nuestra pregunta, los valores de `PRECIO` y `CANTIDAD` son los mas importantes porque nos van a revelar algo clave sobre la inflacion.

In [ ]:
print('=== ESTADISTICAS DESCRIPTIVAS ===')
display(df[['PRECIO', 'CANTIDAD']].describe().round(2))

print('\n--- Interpretacion ---')
print(f'PRECIO minimo:   $ {df["PRECIO"].min():>12,.2f}')
print(f'PRECIO maximo:   $ {df["PRECIO"].max():>12,.2f}')
print(f'PRECIO promedio: $ {df["PRECIO"].mean():>12,.2f}')
print(f'PRECIO mediana:  $ {df["PRECIO"].median():>12,.2f}')
print()
print(f'CANTIDAD minima:   {df["CANTIDAD"].min():>12,.0f}  <- negativa = devolucion')
print(f'CANTIDAD maxima:   {df["CANTIDAD"].max():>12,.0f}')
print(f'CANTIDAD promedio: {df["CANTIDAD"].mean():>12,.0f}')

## 1.7 — Calidad de datos: nulos y duplicados

### ¿Por que revisamos la calidad antes de analizar?

Tres tipos de problemas pueden arruinar un analisis:

1. **Valores nulos**: celdas vacias. Si `CANTIDAD` es nula en un registro, no podemos incluirlo en el analisis.

2. **Filas duplicadas**: el mismo registro aparece dos veces. Inflan los totales artificialmente.

3. **Transacciones de distinto tipo**: en este dataset no todos los registros son ventas reales. La columna `FORMULARIO` diferencia:
   - **FCD** (Factura de Venta): ventas reales → **las que analizamos**
   - **NCD** (Nota de Credito): devoluciones → tienen cantidad negativa, las excluimos
   - **FCR** (Factura Rectificativa): correcciones de facturas → las excluimos

In [ ]:
print('=== VALORES NULOS EN COLUMNAS CLAVE ===')
nulos = df[COLS].isnull().sum()
nulos_relevantes = nulos[nulos > 0]
if len(nulos_relevantes) == 0:
    print('Sin valores nulos en las columnas clave.')
else:
    print(nulos_relevantes)
print()

print('=== FILAS DUPLICADAS ===')
duplicados = df.duplicated().sum()
print(f'Filas duplicadas: {duplicados}')
if duplicados == 0:
    print('Sin duplicados.')
print()

print('=== TIPOS DE TRANSACCION (columna FORMULARIO) ===')
print(df['FORMULARIO'].value_counts())
print()
print('FCD = Factura de Venta  -> ventas reales: LAS QUE ANALIZAMOS')
print('NCD = Nota de Credito   -> devoluciones (cantidad negativa): EXCLUIR')
print('FCR = Factura Rectif.   -> correcciones: EXCLUIR')
print()

print('=== CANTIDADES NEGATIVAS ===')
neg = (df['CANTIDAD'] < 0).sum()
print(f'Registros con cantidad negativa: {neg:,}')
print(f'Estos {neg:,} registros son exactamente las NCD (devoluciones).')

## 1.8 — Exploracion de productos: `.value_counts()`

### ¿Que productos vende General Cereals?

La columna `SUBRUBRO_BI` clasifica cada transaccion segun el tipo de producto. Esta columna es **la mas importante para nuestra pregunta** porque en la seccion 3.3 la vamos a usar para crear la variable `OCTOGONO`: determinar cuales productos llevan el sello de advertencia y cuales no.

`.value_counts()` cuenta cuantas veces aparece cada valor unico en una columna. Es ideal para entender la distribucion de variables categoricas.

In [ ]:
print('=== PRODUCTOS DE GENERAL CEREALS (SUBRUBRO_BI) ===')
print(df['SUBRUBRO_BI'].value_counts())
print()

print('=== PRECLASIFICACION SEGUN LEY 27.642 ===')
print()
CON_OCT = ['COPOS','BOLITAS','ANILLOS','ALMOHADITAS','COPITAS','OSITOS','CARAMELITOS','HONEY GRAHAM','HONEY NUT']
SIN_OCT = ['AVENA','GRANOLA','BRAN','CRISPIES','COPO INTEGRAL','BARRAS DE CEREAL']
print('CON OCTOGONO (cereales azucarados):')
for p in CON_OCT:
    n = (df['SUBRUBRO_BI'] == p).sum()
    if n > 0: print(f'  {p:<20}: {n:>6,} registros')
print()
print('SIN OCTOGONO (cereales naturales):')
for p in SIN_OCT:
    n = (df['SUBRUBRO_BI'] == p).sum()
    if n > 0: print(f'  {p:<20}: {n:>6,} registros')
print()
print('Esta clasificacion se formalizara como la variable OCTOGONO en la seccion 3.3.')

## 1.9 — Cobertura temporal del dataset

### ¿Por que la cobertura temporal es critica para nuestra pregunta?

Toda nuestra pregunta depende de poder comparar **antes y despues** de julio de 2022, cuando entro en vigor la Ley 27.642. Si el dataset no tuviera datos previos a esa fecha, no podriamos aislar el efecto de la ley.

Ademas, hay un detalle importante que vamos a tener que resolver en la seccion 3.3: el periodo **pre-ley** cubre solo 6 meses (enero-junio 2022), mientras que el periodo **post-ley** cubre 30 meses (julio 2022 - diciembre 2024). Si comparamos totales directamente, el post-ley va a parecer siempre mayor simplemente porque tiene mas tiempo. En la seccion 3.3 resolveremos esto usando **promedios mensuales**.

In [ ]:
print('=== COBERTURA TEMPORAL ===')
print(f'Fecha mas antigua: {df["FECHA"].min().date()}')
print(f'Fecha mas reciente: {df["FECHA"].max().date()}')
print(f'Duracion total: {(df["FECHA"].max() - df["FECHA"].min()).days} dias')
print()

print('Transacciones por año:')
print(df['AÑO'].value_counts().sort_index())
print()

LEY_DATE = pd.Timestamp('2022-07-01')
df_fcd = df[df['FORMULARIO'] == 'FCD'].copy()

meses_pre  = df_fcd[df_fcd['FECHA'] < LEY_DATE]['AÑO_MES'].nunique()
meses_post = df_fcd[df_fcd['FECHA'] >= LEY_DATE]['AÑO_MES'].nunique()

print('=== PERIODOS ANTES Y DESPUES DE LA LEY 27.642 ===')
print(f'Fecha de entrada en vigor: 01/07/2022')
print()
print(f'Meses PRE-LEY  (ene-jun 2022):         {meses_pre} meses')
print(f'Meses POST-LEY (jul 2022 - dic 2024):  {meses_post} meses')
print()
print('ATENCION: el periodo post-ley es 5 veces mas largo que el pre-ley.')
print('Comparar totales seria injusto. Usaremos promedios mensuales.')

## 1.10 — Primer grafico: distribucion de ventas por categoria

### ¿Por que un grafico tan temprano?

Un grafico nos da intuicion inmediata sobre los datos. Antes de hacer transformaciones o filtrados, queremos ver como se distribuyen las ventas entre las diferentes categorias de productos. Esto nos permite identificar que categorias son las mas voluminosas y confirmar que hay datos suficientes en ambos grupos (con y sin octogono) para el analisis.

In [ ]:
df_fcd_preview = df[(df['FORMULARIO'] == 'FCD') & (df['CANTIDAD'] > 0)].copy()

ventas_por_categoria = df_fcd_preview.groupby('SUBRUBRO_BI')['CANTIDAD'].sum().sort_values(ascending=True)

CON_OCT_tmp = ['COPOS','BOLITAS','ANILLOS','ALMOHADITAS','COPITAS','OSITOS','CARAMELITOS','HONEY GRAHAM','HONEY NUT']
colores = ['#2171B5' if cat in CON_OCT_tmp else '#6BAED6' for cat in ventas_por_categoria.index]

fig, ax = plt.subplots(figsize=(10, 7))
ventas_por_categoria.plot(kind='barh', ax=ax, color=colores, edgecolor='white', linewidth=0.5)

ax.set_title('Unidades vendidas totales por categoria de producto',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Unidades vendidas', fontsize=11)
ax.set_ylabel('')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.xaxis.grid(True, linestyle='--', alpha=0.7)
ax.yaxis.grid(False)

for spine in ax.spines.values():
    spine.set_visible(False)
ax.spines['bottom'].set_visible(True)
ax.spines['bottom'].set_color('gray')
ax.spines['bottom'].set_linewidth(0.5)

leyenda = [Patch(facecolor='#2171B5', edgecolor='white', label='Con octogono'),
           Patch(facecolor='#6BAED6', edgecolor='white', label='Sin octogono')]
ax.legend(handles=leyenda, loc='lower right', frameon=False)

plt.tight_layout()
plt.show()
print('Grafico 1.10: El volumen de ventas esta concentrado en pocas categorias.')
print('Las categorias CON octogono (azul oscuro) dominan en cantidad de unidades.')

## 1.11 — Sintesis: ¿que encontramos y que hacemos con ello?

Completada la exploracion inicial, tenemos un panorama claro del dataset de General Cereals. Cada hallazgo tiene una implicancia directa para lo que sigue:

| Hallazgo | Implicancia |
|---|---|
| 54.769 filas, 0 duplicados | Dataset limpio, no requiere deduplicacion |
| Solo 2 nulos en CANTIDAD | Se eliminan solos con el filtro de ventas validas |
| 2.237 NCD con cantidad negativa | Filtrar: `FORMULARIO == 'FCD'` y `CANTIDAD > 0` |
| PRE-LEY = 6 meses vs POST-LEY = 30 meses | Comparar **promedios mensuales**, no totales |
| Precios con rango enorme por inflacion | Analizar **cantidades**, no precios ni ingresos |
| 17 categorias claras en SUBRUBRO_BI | Permite crear variable `OCTOGONO` en seccion 3.3 |
| 81% de registros son marca NUTRI FOODS | Datos representativos de General Cereals |

**El dataset es apto para responder la pregunta de investigacion.**

---
# Seccion 3.2 — Pregunta de investigacion

### ¿Que hacemos en esta seccion?

Formulamos de forma precisa la pregunta que queremos responder, definimos las variables del analisis y explicamos el diseno experimental que nos permite aislar el efecto del etiquetado frontal.

## 3.2.1 — Formulacion de la pregunta

> **¿Los productos con octogonos de advertencia (Ley 27.642) experimentaron una mayor caida en las unidades vendidas mensuales promedio respecto de los productos sin octogonos, en el periodo julio 2022 - diciembre 2024 comparado con enero - junio 2022?**

## 3.2.2 — Variables del analisis

| Variable | Tipo | Definicion |
|----------|------|------------|
| **CANTIDAD** | Dependiente (numerica) | Unidades vendidas por transaccion |
| **OCTOGONO** | Independiente (binaria) | 1 = producto con octogono, 0 = sin octogono |
| **PERIODO** | Independiente (binaria) | 1 = post-ley (jul 2022 - dic 2024), 0 = pre-ley (ene - jun 2022) |
| **OCTOGONO x PERIODO** | Interaccion | El efecto causal que buscamos estimar |

La variable **OCTOGONO** se construye a partir de `SUBRUBRO_BI` en la seccion 3.3. La clasificacion se basa en los criterios de la Ley 27.642: productos con exceso de azucares, sodio o grasas reciben el sello de advertencia.

## 3.2.3 — Diseno experimental: Diferencia en Diferencias

### ¿Por que no podemos simplemente comparar ventas antes y despues?

Si miramos solamente las ventas de productos CON octogono antes y despues de la ley, veriamos una caida. Pero esa caida podria deberse a la inflacion, no al octogono. Necesitamos un **grupo de control**.

El grupo de control son los productos **SIN octogono** de la misma empresa. Si ambos grupos sufrieron la misma inflacion pero uno cayo mas que el otro, esa diferencia es atribuible al etiquetado.

El diseno **diferencia en diferencias** compara:

```
Efecto = (Post_CON - Pre_CON) - (Post_SIN - Pre_SIN)
```

Si este valor es **negativo**, significa que los productos CON octogono cayeron mas que los SIN octogono, lo cual sugiere un efecto del etiquetado.

## 3.2.4 — Justificacion de General Cereals como fuente

General Cereals es ideal para este analisis porque:

1. **Misma empresa**: ambos grupos de productos comparten la misma politica de precios, distribucion y marketing.
2. **Mismo mercado**: se venden en los mismos canales y regiones.

Esto reduce la posibilidad de que factores externos expliquen la diferencia entre grupos.

## 3.2.5 — Limitaciones del analisis

| Limitacion | Impacto | Mitigacion |
|------------|---------|------------|
| Periodos desiguales (6 vs 30 meses) | Los totales no son comparables | Usar promedios mensuales |
| Datos de una sola empresa | No generalizable a todo el mercado | Interpretar como caso especifico |
| Solo vendas de mayorsita | No refleja consumo final | Los datos son de un solo canal B2B |
| Posible sesgo estacional | Las ventas varian segun la epoca del año | El pre-ley cubre solo 6 meses, posiblemente no representativo |
| No hay precio de productos sin octogono equivalentes | No se puede controlar por precio | Enfocarse en cantidades, no en precios |

## 3.2.6 — Hipotesis

- **H0 (nula)**: No hay diferencia en la variacion de ventas mensuales entre productos con y sin octogono despues de la ley.
- **H1 (alternativa)**: Los productos con octogono experimentaron una mayor caida en ventas mensuales que los productos sin octogono despues de la ley.

Rechazaremos H0 si el coeficiente de interaccion OCTOGONO x PERIODO es **negativo y estadisticamente significativo** (p-value < 0.05).

---
# Seccion 3.3 — Transformaciones y resumen estadistico

### ¿Que hacemos en esta seccion?

Aplicamos todas las transformaciones identificadas en la seccion 3.1:
1. Filtrar solo ventas reales (FCD con CANTIDAD > 0)
2. Crear la variable `OCTOGONO` (binaria)
3. Crear la variable `PERIODO` (pre-ley / post-ley)
4. Agrupar por mes y calcular promedios
5. Generar estadisticas resumen por grupo

In [ ]:
df_fcd = df[(df['FORMULARIO'] == 'FCD') & (df['CANTIDAD'] > 0)].copy()
df_fcd.reset_index(drop=True, inplace=True)

print(f'Despues del filtro:')
print(f'  Filas: {df_fcd.shape[0]:,}  (se excluyeron {df.shape[0] - df_fcd.shape[0]:,} registros)')
print(f'  CANTIDAD minima: {df_fcd["CANTIDAD"].min():,.0f} (todas positivas)')

In [ ]:
# CON_OCT y SIN_OCT ya estan definidos en la seccion 1.8

df_fcd['OCTOGONO'] = df_fcd['SUBRUBRO_BI'].apply(lambda x: 1 if x in CON_OCT else 0)

LEY_DATE = pd.Timestamp('2022-07-01')
df_fcd['PERIODO'] = df_fcd['FECHA'].apply(lambda x: 'Post-ley' if x >= LEY_DATE else 'Pre-ley')

print('=== VARIABLES CREADAS ===')
print(f'OCTOGONO: {(df_fcd["OCTOGONO"] == 1).sum():,} registros con octogono, {(df_fcd["OCTOGONO"] == 0).sum():,} sin octogono')
print(f'PERIODO:  {(df_fcd["PERIODO"] == "Post-ley").sum():,} registros post-ley, {(df_fcd["PERIODO"] == "Pre-ley").sum():,} pre-ley')
print()
print('Distribucion cruzada:')
print(df_fcd.groupby(['PERIODO', 'OCTOGONO']).size().unstack(fill_value=0))

In [ ]:
ventas_mensuales = df_fcd.groupby(['AÑO_MES', 'OCTOGONO', 'PERIODO']).agg(
    CANTIDAD_TOTAL=('CANTIDAD', 'sum'),
    N_TRANSACCIONES=('CANTIDAD', 'count')
).reset_index()

print('=== VENTAS MENSUALES POR GRUPO ===')
display(ventas_mensuales.head(10))
print(f'\nTotal de registros mensuales: {ventas_mensuales.shape[0]}')

In [ ]:
resumen = ventas_mensuales.groupby(['PERIODO', 'OCTOGONO']).agg(
    PROMEDIO_MENSUAL=('CANTIDAD_TOTAL', 'mean'),
    DESVIACION=('CANTIDAD_TOTAL', 'std'),
    MESES=('CANTIDAD_TOTAL', 'count')
).round(0)

resumen.index = resumen.index.set_levels(['Sin octogono', 'Con octogono'], level=1)
resumen.index = resumen.index.set_levels(['Post-ley', 'Pre-ley'], level=0)

print('=== RESUMEN: PROMEDIO MENSUAL DE UNIDADES VENDIDAS ===')
display(resumen)

con_pre = resumen.loc[('Pre-ley', 'Con octogono'), 'PROMEDIO_MENSUAL']
con_post = resumen.loc[('Post-ley', 'Con octogono'), 'PROMEDIO_MENSUAL']
sin_pre = resumen.loc[('Pre-ley', 'Sin octogono'), 'PROMEDIO_MENSUAL']
sin_post = resumen.loc[('Post-ley', 'Sin octogono'), 'PROMEDIO_MENSUAL']

var_con = ((con_post - con_pre) / con_pre) * 100
var_sin = ((sin_post - sin_pre) / sin_pre) * 100

print(f'\n--- Variacion porcentual pre -> post ---')
print(f'CON octogono:    {var_con:+.1f}%')
print(f'SIN octogono:    {var_sin:+.1f}%')
print(f'Diferencia:      {var_con - var_sin:+.1f} puntos porcentuales')

---
# Seccion 3.4 — Analisis grafico

### ¿Que hacemos en esta seccion?

Visualizamos los resultados de la seccion 3.3 para comunicar los hallazgos de forma clara. Usamos principios de Edward Tufte: maxima informacion con minima tinta, paletas tonales y sin ruido visual.

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Blues")

COLOR_CON = '#2171B5'
COLOR_SIN = '#6BAED6'

def clean_spines(ax, keep=['bottom', 'left']):
    for spine in ax.spines.values():
        spine.set_visible(False)
    for spine in keep:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_color('gray')
        ax.spines[spine].set_linewidth(0.5)

print('Configuracion grafica aplicada.')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

categorias = ['Pre-ley\nSin octogono', 'Pre-ley\nCon octogono',
              'Post-ley\nSin octogono', 'Post-ley\nCon octogono']
valores = [sin_pre, con_pre, sin_post, con_post]
colores = [COLOR_SIN, COLOR_CON, COLOR_SIN, COLOR_CON]

barras = ax.bar(categorias, valores, color=colores, edgecolor='white', linewidth=1.5, width=0.6)

for barra, valor in zip(barras, valores):
    ax.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 500,
            f'{valor:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

clean_spines(ax)
ax.set_title('Promedio mensual de unidades vendidas\npor periodo y tipo de producto',
             fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Unidades vendidas (promedio mensual)', fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.xaxis.grid(False)

leyenda = [Patch(facecolor=COLOR_CON, edgecolor='white', label='Con octogono'),
           Patch(facecolor=COLOR_SIN, edgecolor='white', label='Sin octogono')]
ax.legend(handles=leyenda, loc='upper right', frameon=False)

plt.tight_layout()
plt.show()
print('Grafico 3.4.1: Comparacion de promedios mensuales pre vs post por grupo.')

In [ ]:
ventas_mensuales['OCTOGONO_LABEL'] = ventas_mensuales['OCTOGONO'].map({1: 'Con octogono', 0: 'Sin octogono'})

fig, ax = plt.subplots(figsize=(14, 6))

for label, color in [('Con octogono', COLOR_CON), ('Sin octogono', COLOR_SIN)]:
    subset = ventas_mensuales[ventas_mensuales['OCTOGONO_LABEL'] == label]
    meses_num = [str(m) for m in subset['AÑO_MES']]
    ax.plot(meses_num, subset['CANTIDAD_TOTAL'], marker='o', markersize=4,
            label=label, color=color, linewidth=2)

ax.axvline(x=5, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='Ley 27.642 (jul 2022)')

clean_spines(ax)
ax.set_title('Evolucion mensual de unidades vendidas por grupo',
             fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Unidades vendidas', fontsize=11)
ax.set_xlabel('')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.xaxis.grid(False)

ticks_idx = list(range(0, len(meses_num), 3))
ax.set_xticks([meses_num[i] for i in ticks_idx])
ax.set_xticklabels([meses_num[i] for i in ticks_idx], rotation=45, ha='right')

ax.legend(loc='upper right', frameon=False)
plt.tight_layout()
plt.show()
print('Grafico 3.4.2: Evolucion temporal con linea de corte en la entrada en vigor de la ley.')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

data_box = [ventas_mensuales[(ventas_mensuales['PERIODO'] == 'Pre-ley') & (ventas_mensuales['OCTOGONO'] == 0)]['CANTIDAD_TOTAL'],
            ventas_mensuales[(ventas_mensuales['PERIODO'] == 'Pre-ley') & (ventas_mensuales['OCTOGONO'] == 1)]['CANTIDAD_TOTAL'],
            ventas_mensuales[(ventas_mensuales['PERIODO'] == 'Post-ley') & (ventas_mensuales['OCTOGONO'] == 0)]['CANTIDAD_TOTAL'],
            ventas_mensuales[(ventas_mensuales['PERIODO'] == 'Post-ley') & (ventas_mensuales['OCTOGONO'] == 1)]['CANTIDAD_TOTAL']]

bp = ax.boxplot(data_box, patch_artist=True, widths=0.5)

colores_box = [COLOR_SIN, COLOR_CON, COLOR_SIN, COLOR_CON]
for patch, color in zip(bp['boxes'], colores_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xticklabels(['Pre-ley\nSin octogono', 'Pre-ley\nCon octogono',
                     'Post-ley\nSin octogono', 'Post-ley\nCon octogono'])

clean_spines(ax)
ax.set_title('Distribucion de ventas mensuales por grupo',
             fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Unidades vendidas', fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.xaxis.grid(False)

plt.tight_layout()
plt.show()
print('Grafico 3.4.3: Boxplot de la dispersion de ventas mensuales por grupo.')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

grupos = ['Sin octogono', 'Con octogono']
variaciones = [var_sin, var_con]
colores_var = [COLOR_SIN, COLOR_CON]

barras = ax.barh(grupos, variaciones, color=colores_var, edgecolor='white', linewidth=1.5, height=0.5)

for barra, valor in zip(barras, variaciones):
    pos_x = valor - 3 if valor < 0 else valor + 1
    ax.text(pos_x, barra.get_y() + barra.get_height()/2,
            f'{valor:+.1f}%', ha='center', va='center', fontsize=12, fontweight='bold')

ax.axvline(x=0, color='gray', linewidth=0.8)
clean_spines(ax)
ax.set_title('Variacion porcentual de ventas mensuales\npre-ley a post-ley',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Variacion (%)', fontsize=11)
ax.xaxis.grid(True, linestyle='--', alpha=0.7)
ax.yaxis.grid(False)

plt.tight_layout()
plt.show()
print(f'Grafico 3.4.4: La diferencia de {var_con - var_sin:+.1f} pp sugiere efecto del etiquetado.')

---
# Seccion 3.5 — Aplicacion de conceptos de la materia

### ¿Que hacemos en esta seccion?

Aplicamos los conceptos de Metodos Cuantitativos vistos en la materia para formalizar estadisticamente el hallazgo de las secciones anteriores.

## 3.5.1 — Distribucion de probabilidad

Analizamos la forma de la distribucion de ventas mensuales para cada grupo. Esto nos dice si los datos cumplen las suposiciones necesarias para aplicar tecnicas estadisticas como la regresion lineal.

*Conceptos aplicados: medidas de distribucion y frecuencias (Clase 2 - Manipulacion de datos con Pandas y NumPy).*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, oct_val, label, color in zip(axes, [0, 1], ['Sin octogono', 'Con octogono'], [COLOR_SIN, COLOR_CON]):
    subset = ventas_mensuales[ventas_mensuales['OCTOGONO'] == oct_val]['CANTIDAD_TOTAL']
    ax.hist(subset, bins=8, color=color, edgecolor='white', linewidth=0.5, alpha=0.8)
    ax.axvline(subset.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Media: {subset.mean():,.0f}')
    ax.axvline(subset.median(), color='darkblue', linestyle=':', linewidth=1.5, label=f'Mediana: {subset.median():,.0f}')
    clean_spines(ax)
    ax.set_title(f'Distribucion de ventas mensuales\n{label}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Unidades vendidas', fontsize=10)
    ax.set_ylabel('Frecuencia', fontsize=10)
    ax.legend(fontsize=9, frameon=False)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.show()

print('--- Estadisticas de distribucion ---')
for oct_val, label in [(0, 'Sin octogono'), (1, 'Con octogono')]:
    s = ventas_mensuales[ventas_mensuales['OCTOGONO'] == oct_val]['CANTIDAD_TOTAL']
    print(f'{label}: media={s.mean():,.0f}, mediana={s.median():,.0f}, std={s.std():,.0f}, CV={s.std()/s.mean()*100:.1f}%')

## 3.5.2 — Medidas de tendencia central y dispersion

| Medida | Sin octogono pre | Con octogono pre | Sin octogono post | Con octogono post |
|--------|-----------------|-----------------|-------------------|-------------------|
| Media | Ver calculo abajo | Ver calculo abajo | Ver calculo abajo | Ver calculo abajo |
| Mediana | Ver calculo abajo | Ver calculo abajo | Ver calculo abajo | Ver calculo abajo |
| Desv. estandar | Ver calculo abajo | Ver calculo abajo | Ver calculo abajo | Ver calculo abajo |

El **coeficiente de variacion** (CV = desviacion estandar / media) nos indica que tan dispersos estan los datos. Un CV alto indica alta variabilidad mensual.

*Conceptos aplicados: tendencia central y dispersion (Clase 2), estadisticas descriptivas con `.describe()` y `.groupby()`.*

In [ ]:
print('=== MEDIDAS DE TENDENCIA CENTRAL Y DISPERSION ===')
print()
for periodo in ['Pre-ley', 'Post-ley']:
    print(f'--- {periodo} ---')
    for oct_val, label in [(0, 'Sin octogono'), (1, 'Con octogono')]:
        s = ventas_mensuales[(ventas_mensuales['PERIODO'] == periodo) & (ventas_mensuales['OCTOGONO'] == oct_val)]['CANTIDAD_TOTAL']
        cv = (s.std() / s.mean() * 100) if s.mean() != 0 else 0
        print(f'  {label:>15}: media={s.mean():>10,.0f} | mediana={s.median():>10,.0f} | std={s.std():>10,.0f} | CV={cv:.1f}%')
    print()

## 3.5.3 — Regresion lineal multiple

### Modelo estadistico

Estimamos el siguiente modelo de regresion lineal:

$$CANTIDAD = \beta_0 + \beta_1 \cdot OCTOGONO + \beta_2 \cdot POST + \beta_3 \cdot (OCTOGONO \times POST) + \epsilon$$

Donde:
- $\beta_0$: ventas base (pre-ley, sin octogono)
- $\beta_1$: diferencia entre con y sin octogono en pre-ley
- $\beta_2$: efecto de la ley en productos sin octogono
- $\beta_3$: **efecto causal del etiquetado** — la diferencia adicional que sufrieron los productos con octogono

Si $\beta_3 < 0$ y es significativo, hay evidencia de que el etiquetado redujo las ventas.

*Conceptos aplicados: algebra lineal — ecuaciones normales con inversa de matriz (Clase 5-6: Matrices y DataFrames). Los coeficientes se estiman resolviendo $\beta = (X^T X)^{-1} X^T y$ usando `numpy.linalg`.*

In [ ]:
df_reg = ventas_mensuales.copy()
df_reg['POST'] = (df_reg['PERIODO'] == 'Post-ley').astype(int)
df_reg['INTERACCION'] = df_reg['OCTOGONO'] * df_reg['POST']

# --- Modelo simbolico con sympy ---
beta_0, beta_1, beta_2, beta_3 = sp.symbols('beta_0 beta_1 beta_2 beta_3', real=True)
OCT, POST, INTER = sp.symbols('OCT POST INTER', real=True)

CANTIDAD_modelo = beta_0 + beta_1 * OCT + beta_2 * POST + beta_3 * INTER
print('=== MODELO DE REGRESION LINEAL ===')
print()
print('Ecuacion del modelo (simbolica):')
print(f'  CANTIDAD = {CANTIDAD_modelo}')
print()
print('Donde:')
print('  beta_0 = intercepto (ventas base: pre-ley, sin octogono)')
print('  beta_1 = efecto del octogono en pre-ley')
print('  beta_2 = efecto de la ley en productos sin octogono')
print('  beta_3 = efecto causal del etiquetado (interaccion)')
print()

# --- Estimacion con numpy (ecuaciones normales) ---
X = df_reg[['OCTOGONO', 'POST', 'INTERACCION']].values
y = df_reg['CANTIDAD_TOTAL'].values

X_with_const = np.column_stack([np.ones(len(X)), X])
beta_hat = np.linalg.lstsq(X_with_const, y, rcond=None)[0]

y_pred = X_with_const @ beta_hat
ss_res = np.sum((y - y_pred) ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)
r2 = 1 - ss_res / ss_tot

n = len(y)
p = X_with_const.shape[1]
r2_ajustado = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print('--- Coeficientes estimados (ecuaciones normales) ---')
print(f'Intercepto (beta_0):          {beta_hat[0]:>12,.0f}  (ventas base: pre-ley, sin octogono)')
print(f'OCTOGONO (beta_1):            {beta_hat[1]:>12,.0f}  (diferencia con/sin en pre-ley)')
print(f'POST (beta_2):                {beta_hat[2]:>12,.0f}  (efecto ley en sin octogono)')
print(f'INTERACCION (beta_3):         {beta_hat[3]:>12,.0f}  (efecto causal del etiquetado)')
print()
print(f'R-cuadrado:                   {r2:>12.4f}')
print(f'R-cuadrado ajustado:          {r2_ajustado:>12.4f}')
print()

# --- Ecuacion con valores numericos ---
CANTIDAD_numerico = beta_hat[0] + beta_hat[1]*OCT + beta_hat[2]*POST + beta_hat[3]*INTER
print('Ecuacion con coeficientes estimados:')
print(f'  CANTIDAD = {beta_hat[0]:,.0f} + ({beta_hat[1]:,.0f})*OCT + ({beta_hat[2]:,.0f})*POST + ({beta_hat[3]:,.0f})*INTER')
print()

print('--- Interpretacion ---')
print(f'El modelo explica el {r2*100:.1f}% de la varianza en las ventas mensuales.')
if r2 >= 0.8:
    print('Ajuste EXCELENTE.')
elif r2 >= 0.6:
    print('Buen ajuste.')
elif r2 >= 0.4:
    print('Ajuste moderado.')
else:
    print('Ajuste debil — hay mucha variabilidad no explicada por el modelo.')

In [ ]:
import math
residuos = y - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_pred, residuos, color=COLOR_CON, edgecolor='white', linewidth=0.5, s=60)
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=1)
clean_spines(axes[0])
axes[0].set_title('Residuos vs Valores predichos', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Valores predichos', fontsize=10)
axes[0].set_ylabel('Residuos', fontsize=10)

# Q-Q plot: cuantiles observados vs teoricos (normal estandar)
residuos_sorted = np.sort(residuos)
n_res = len(residuos_sorted)
probabilidades = (np.arange(1, n_res + 1) - 0.5) / n_res
cuantiles_teoricos = np.sqrt(2) * np.array([math.erfcinv(2 * (1 - p)) for p in probabilidades])

axes[1].scatter(cuantiles_teoricos, residuos_sorted, color=COLOR_CON, edgecolor='white', linewidth=0.5, s=60)
lim_min = min(cuantiles_teoricos.min(), residuos_sorted.min())
lim_max = max(cuantiles_teoricos.max(), residuos_sorted.max())
axes[1].plot([lim_min, lim_max], [lim_min, lim_max], 'r--', linewidth=1)
clean_spines(axes[1])
axes[1].set_title('Q-Q Plot (Normalidad de residuos)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Cuantiles teoricos', fontsize=10)
axes[1].set_ylabel('Cuantiles observados', fontsize=10)

plt.tight_layout()
plt.show()

skewness = np.mean(((residuos - residuos.mean()) / residuos.std()) ** 3)
kurtosis = np.mean(((residuos - residuos.mean()) / residuos.std()) ** 4) - 3
print(f'Sesgo (skewness):  {skewness:.3f}')
print(f'Curtosis:          {kurtosis:.3f}')
if abs(skewness) < 1:
    print('Los residuos son aproximadamente simetricos.')
else:
    print('ATENCION: los residuos tienen sesgo considerable.')

## 3.5.4 — Significancia estadistica

Para determinar si el coeficiente de interaccion (beta_3) es estadisticamente significativo, calculamos el **p-value** usando una prueba t.

- Si **p-value < 0.05**: rechazamos H0 → el efecto del etiquetado es significativo.
- Si **p-value >= 0.05**: no podemos rechazar H0 → no hay evidencia suficiente.

*Conceptos aplicados: errores estandar de coeficientes, estadistico t y distribucion normal (Clase 9-10: Calculo diferencial y aplicaciones economicas). Se calcula manualmente con `numpy.linalg.inv` para la varianza de los coeficientes.*

In [ ]:
# --- Prueba de significancia con numpy ---
# Calculamos errores estandar, t-estadisticos y p-values manualmente

import math

X_with_const2 = np.column_stack([np.ones(len(X)), X])
beta_ls = np.linalg.lstsq(X_with_const2, y, rcond=None)[0]
y_pred_ls = X_with_const2 @ beta_ls

sse = np.sum((y - y_pred_ls) ** 2)
n_obs = len(y)
k = X_with_const2.shape[1]
mse = sse / (n_obs - k)

var_beta = mse * np.linalg.inv(X_with_const2.T @ X_with_const2).diagonal()
se_beta = np.sqrt(var_beta)

t_stats = beta_ls / se_beta

# CDF de la normal estandar usando math.erf (estandar de Python)
def normal_cdf(x):
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

p_values = np.array([2 * (1 - normal_cdf(abs(t))) for t in t_stats])

nombres = ['Intercepto', 'OCTOGONO', 'POST', 'INTERACCION']

print('=== PRUEBA DE SIGNIFICANCIA ===')
print()
print(f'{"Variable":<15} {"Coef":>12} {"Error Est":>12} {"t-estadistico":>14} {"p-value":>10} {"Significativo?":>15}')
print('-' * 80)
for nombre, b, s, t, p in zip(nombres, beta_ls, se_beta, t_stats, p_values):
    sig = '*** SI ***' if p < 0.05 else 'No'
    print(f'{nombre:<15} {b:>12,.0f} {s:>12,.0f} {t:>14.3f} {p:>10.4f} {sig:>15}')

print()
beta3 = beta_ls[3]
p3 = p_values[3]
print(f'CONCLUSION: beta_3 = {beta3:,.0f} con p-value = {p3:.4f}')
if p3 < 0.05 and beta3 < 0:
    print('Hay evidencia estadisticamente significativa de que el etiquetado REDUJO las ventas de productos con octogono.')
elif p3 < 0.05 and beta3 > 0:
    print('El efecto es significativo pero positivo: los productos con octogono vendieron MAS.')
else:
    print('NO hay evidencia estadisticamente significativa del efecto del etiquetado.')

## 3.5.5 — Conclusion general

### Sintesis de hallazgos

| Hallazgo | Valor |
|----------|-------|
| Variacion ventas CON octogono (pre a post) | Ver seccion 3.3 |
| Variacion ventas SIN octogono (pre a post) | Ver seccion 3.3 |
| Diferencia atribuible al etiquetado | Ver seccion 3.3 |
| R-cuadrado del modelo | Ver seccion 3.5.3 |
| Significancia del efecto (p-value) | Ver seccion 3.5.4 |

### Respuesta a la pregunta de investigacion

Basandonos en el analisis de General Cereals, los datos sugieren que **[completar segun resultados]**:

- Si beta_3 < 0 y p < 0.05: los productos con octogono experimentaron una caida adicional en ventas, atribuible al etiquetado frontal.
- Si beta_3 no es significativo: no hay evidencia suficiente para afirmar que el etiquetado afecto las ventas de manera diferencial.

### Limitaciones finales

1. Los datos son solo de una empresa (General Cereals), por lo que no son representativos de todo el mercado.
2. El periodo pre-ley es corto (6 meses), lo que puede no capturar la variabilidad estacional completa.
3. No contamos con datos de precios de consumo final ni de la competencia.
4. El modelo asume linealidad y no controla por otros factores macroeconomicos.